# Phase 4: Feature Engineering

This notebook documents the Phase 4 feature engineering workflow for the Bosch Production Line Performance project.

Phase 4 objectives:

19. Create `start_time` and `end_time` features.
20. Create `cycle_time` and `processing_duration` features.
21. Create `waiting_time` and delay metrics.
22. Generate station count features.
23. Generate product path complexity scores.
24. Create aggregated statistics for each production line.

All features are engineered directly from the raw date CSV files. `Response` is joined from `train_numeric.csv` for train rows only.

## 1. Import Phase 4 Helpers

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.data.phase4_feature_engineering import (
    REPORTS_DIR,
    PROCESSED_DIR,
    build_date_column_groups,
    create_feature_dictionary,
    engineer_split,
    summarize_engineered_features,
    write_report,
)

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 180)

print('Project root:', PROJECT_ROOT)

## 2. Inspect Date Feature Groups

The raw date columns are grouped by station and production line. These groups drive the station-level and line-level feature calculations.

In [ ]:
station_columns, line_columns = build_date_column_groups()

print('Stations:', len(station_columns))
print('Lines:', len(line_columns))
print('First stations:', list(station_columns.keys())[:10])
print('Line date feature counts:', {line: len(columns) for line, columns in line_columns.items()})

## 3. Feature Definitions

- `start_time`: earliest observed raw date value for a product.
- `end_time`: latest observed raw date value for a product.
- `cycle_time`: `end_time - start_time`.
- `processing_duration`: summed within-station active durations.
- `waiting_time`: sum of positive gaps between consecutive station end/start times.
- `delay_ratio`: `waiting_time / cycle_time` when cycle time is positive.
- `station_count`: number of stations with at least one observed date value.
- `line_count`: number of lines touched by a product.
- `path_density`: station coverage between first and last observed station.
- `line_switch_count`: number of production-line changes along the observed station sequence.
- `path_complexity_score`: composite score using station count, line count, line switches, and path density.
- `line_{n}_...`: line-level timing, station-count, and completeness aggregates.

## 4. Engineer Train and Test Features

This step scans the raw date CSVs in chunks and writes compact engineered CSV files to `data/processed/`.

In [ ]:
train_path = engineer_split('train', chunksize=20_000)
test_path = engineer_split('test', chunksize=20_000)

train_path, test_path

## 5. Preview Engineered Features

In [ ]:
train_preview = pd.read_csv(train_path, nrows=10)
test_preview = pd.read_csv(test_path, nrows=10)

display(train_preview)
display(test_preview)

## 6. Save Feature Dictionary and Report

In [ ]:
feature_dictionary = create_feature_dictionary()
summary = summarize_engineered_features(train_path, test_path)

feature_dictionary.to_csv(REPORTS_DIR / 'phase4_feature_dictionary.csv', index=False)
summary.to_csv(REPORTS_DIR / 'phase4_engineered_feature_summary.csv', index=False)
write_report(summary, feature_dictionary)

display(summary)
display(feature_dictionary)

## 7. Phase 4 Deliverables

After running this notebook, the project has model-ready engineered train/test feature files, a feature dictionary, and a Phase 4 report.